# Stro-7 contrastive cosine: human nuclei and their transfer to mouse

How closely each human fibroblast/decidual nucleus resembles each of the 11 stromal subtypes of Ulrich et al. 2024
(PNAS 121:e2404775121), with Stro-7, the C7-fibroblast, as the subtype of interest; then the same score carried to every
mouse fibroblast/decidual cell through the SAMap graph. The Stro-7 columns are `obs['stro7_cosine']` of the final human
object and `obs['stro7_cosine_transferred']` of the final mouse object, and this notebook checks that it reproduces both.

- **Cosine.** Over the 3,000 genes that vary most across the 11 subtype centroids once every subtype's top 50 markers are
  set aside: the cosine between a nucleus's profile minus the mean human nucleus, and a subtype's centroid minus the mean
  of the 11 centroids.
- **Transfer.** Each mouse cell gets the mean cosine of its human neighbors in the SAMap graph, weighted by the graph's
  connectivities.

The same computation runs as rules `c7_stro7_cosine` and `c7_tables` in `xmens_final`
(`workflow/rules/41_fig_c7_presence.smk`). To run this notebook on the cluster, submit `stro7_cosine.sbatch` from the
folder that should receive the outputs; to run it by hand, open it in Jupyter with a kernel running `rsc_uv`.

In [ ]:
# Parameters; papermill -p NAME VALUE overrides them.

# The SAMap object of record (pinned by sha256 in xmens_final config/config.yaml, samap_figures.joint_sha256)
SAMAP_H5AD = "/n/lab_storage/eddy_lab/users/nhilgert/code/samap-scvi/full_fibdec/runs_raw/vanilla_raw/samap_transferred_pt_harmonized.h5ad"

# Ulrich et al. 2024 Dataset S5 (subtype centroids) and Dataset S6 (subtype markers), as published
ULRICH_S5 = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/external/ulrich2024/pnas.2404775121.sd05.xlsx"
ULRICH_S6 = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/external/ulrich2024/pnas.2404775121.sd06.xlsx"

# The final objects whose stored columns this run must reproduce; None skips that comparison.
HUMAN_OBJECT = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MOUSE_OBJECT = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"

OUT_DIR = None  # folder for the two output tables; None writes nothing

In [ ]:
import hashlib
import platform
from importlib.metadata import version
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from anndata.io import read_elem
from scipy.stats import spearmanr

FIBROBLAST_SUBTYPES = [5, 7, 8, 9, 10, 11]
CONTROL_SUBTYPES = [1, 2, 3, 4, 6]  # perivascular and smooth muscle: subtypes a fibroblast should not resemble
TARGET = "Stro-7"  # C7-fibroblasts
N_MARKERS = 50  # each subtype's top markers by avg_log2FC, set aside from the panel
N_PANEL = 3000  # genes in the panel
TOLERANCE = 1e-5  # largest |recomputed - stored| accepted, as in xmens_final config/c7_presence.yaml


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


def subtype_number(label):
    """7 for 'Stro-7_C7 Fibroblast' (Dataset S5) or '7_C7 Fibroblast' (Dataset S6); None for other labels."""
    head = str(label).split("_")[0].replace("Stro-", "").strip()
    return int(head) if head.isdigit() else None


def read_obs_column(path, column):
    """One obs column of an .h5ad, read with h5py so the expression matrix is never loaded."""
    with h5py.File(path, "r") as handle:
        obs = read_elem(handle["obs"])
    return obs[column].set_axis(obs.index.astype(str))


packages = ", ".join(f"{name} {version(name)}" for name in ("numpy", "pandas", "scipy", "anndata", "h5py"))
print(f"python {platform.python_version()} | {packages}")
for path in (SAMAP_H5AD, ULRICH_S5, ULRICH_S6):
    print(sha256_of(path), path)

## Atlas: subtype centroids, markers and the gene panel

Dataset S5 (sheet 5B) gives each of the atlas's 39 subtypes its mean expression; the notebook uses the 11 stromal ones.
Dataset S6 (sheet 6C) lists each stromal subtype's markers. The top 50 of every subtype are set aside from the panel, so
the score does not rest on the genes that define the subtypes; C7 is one of them.

In [ ]:
centroids = pd.read_excel(ULRICH_S5, sheet_name="5B Subtype centroids", header=2, index_col=0)
centroids.columns = pd.Index(centroids.columns).astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
centroids.index = centroids.index.astype(str).str.strip()

# Rows without a gene symbol are blank spacer rows.
centroids = centroids[~centroids.index.isin(["nan", ""])]
assert centroids.index.is_unique

subtypes = FIBROBLAST_SUBTYPES + CONTROL_SUBTYPES
names = [f"Stro-{n}" for n in subtypes]
column_of = {subtype_number(c): c for c in centroids.columns if c.startswith("Stro-")}
atlas = centroids[[column_of[n] for n in subtypes]].apply(pd.to_numeric).set_axis(names, axis=1)
assert atlas.notna().all().all()

print(f"{len(atlas):,} genes x {atlas.shape[1]} subtypes:", ", ".join(column_of[n] for n in subtypes))

In [ ]:
markers_table = pd.read_excel(ULRICH_S6, sheet_name="6C Stro markers", header=2)
markers_table.columns = ["p_val", "avg_log2FC", "pct.1", "pct.2", "p_val_adj", "cluster", "gene"]
markers_table = markers_table.dropna(subset=["gene", "cluster"])
markers_table["subtype"] = markers_table["cluster"].map(subtype_number)

# Each subtype's genes by decreasing avg_log2FC, repeats dropped, first N_MARKERS kept
markers = {}
for n in subtypes:
    ranked = markers_table[markers_table["subtype"] == n].sort_values("avg_log2FC", ascending=False)["gene"]
    markers[n] = list(dict.fromkeys(ranked.astype(str)))[:N_MARKERS]

print("markers per subtype:", {f"Stro-{n}": len(genes) for n, genes in markers.items()})

## Human nuclei

The SAMap object holds both species: cells carry `species` `hs` or `mm`, and genes are named `hs_<symbol>` or
`mm_<symbol>`. The cosine uses the human nuclei and the human genes, with expression as stored in the object's X
(log scale).

In [ ]:
with h5py.File(SAMAP_H5AD, "r") as handle:
    obs = read_elem(handle["obs"])
    var_names = pd.Index(read_elem(handle["var"]).index.astype(str))
    X = read_elem(handle["X"])
obs.index = obs.index.astype(str)
X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)

# Human genes by symbol (column index into X); a symbol that occurs twice keeps its first column.
is_human_gene = var_names.str.split("_").str[0].isin(["hs", "human"])
gene_column = pd.Series(np.flatnonzero(is_human_gene), index=var_names[is_human_gene].str.split("_", n=1).str[1])
gene_column = gene_column[~gene_column.index.duplicated(keep="first")]

is_human = obs["species"].astype(str).isin(["hs", "human"]).to_numpy()
human_nuclei = obs.index[is_human]
expr = X[np.flatnonzero(is_human)]
del X

# The source script took log1p when X held counts (a maximum above 50); this X is already log scale.
assert expr.max() <= 50, f"X looks like counts (maximum {expr.max():.0f}); this notebook expects log-scale expression"

print(f"SAMap object: {len(obs):,} cells x {len(var_names):,} genes | human nuclei {len(human_nuclei):,} | "
      f"human genes {len(gene_column):,} | maximum of X {expr.max():.2f}")

In [ ]:
# Genes in both the atlas and the SAMap object, ranked by variance across the 11 centroids, markers removed
shared = gene_column.index.intersection(atlas.index)
set_aside = {gene for genes in markers.values() for gene in genes} & set(shared)
variance = atlas.loc[shared].var(axis=1).sort_values(ascending=False)
panel = [gene for gene in variance.index if gene not in set_aside][:N_PANEL]

assert len(panel) == N_PANEL and not set(panel) & set_aside and "C7" not in panel
print(f"shared genes {len(shared):,} | markers set aside {len(set_aside):,} | panel {len(panel):,}")

In [ ]:
# Each side is centered on its own population: the score asks whether a nucleus departs from the average human nucleus
# in the direction a subtype departs from the average stromal subtype.
profile = expr[:, gene_column.loc[panel].to_numpy()].toarray().astype(np.float32)
profile = profile - profile.mean(axis=0, keepdims=True)
length = np.linalg.norm(profile, axis=1, keepdims=True)
length[length == 0] = 1.0  # a nucleus at the exact mean keeps a zero vector instead of dividing by zero
profile = profile / length

direction = atlas.loc[panel].to_numpy(np.float32)
direction = direction - direction.mean(axis=1, keepdims=True)
direction = direction / np.linalg.norm(direction, axis=0, keepdims=True)

cosine = pd.DataFrame(profile @ direction, index=pd.Index(human_nuclei, name="obs_name"), columns=names)
print("median cosine by subtype:", cosine.median().astype(float).round(4).to_dict())

C7 is set aside from the panel, so its expression is an independent read-out of the score: of the 11 subtype cosines,
the Stro-7 cosine should follow it most closely.

In [ ]:
c7 = expr[:, gene_column["C7"]].toarray().ravel()
rho = pd.Series({name: spearmanr(c7, cosine[name]).statistic for name in names}).sort_values(ascending=False)

print(f"C7 detected in {np.mean(c7 > 0):.1%} of human nuclei; Spearman correlation of C7 with each cosine:")
print(rho.round(3).to_string())
assert rho.index[0] == TARGET, f"C7 follows {rho.index[0]} most closely, not {TARGET}"

## Transfer to mouse through the SAMap graph

Each mouse cell takes the mean of its human neighbors' cosines, weighted by the connectivities of the SAMap graph
(`obsp['connectivities']`), the same operator that transfers pseudotime. All human nuclei are sources. A mouse cell
with no human neighbor gets NaN.

In [ ]:
def transfer(graph, values, is_source):
    """Each cell's connectivity-weighted mean of its source neighbors' values; NaN without a source neighbor."""
    has_value = is_source & np.isfinite(values)
    weighted_sum = graph.dot(np.where(has_value, values, 0.0))
    weight = graph.dot(has_value.astype(float))
    mean = np.full(graph.shape[0], np.nan)
    mean[weight > 0] = weighted_sum[weight > 0] / weight[weight > 0]
    return mean


with h5py.File(SAMAP_H5AD, "r") as handle:
    connectivities = read_elem(handle["obsp"]["connectivities"]).tocsr()
assert connectivities.shape == (len(obs), len(obs))

is_mouse = obs["species"].astype(str).isin(["mm", "mouse"]).to_numpy()
transferred = pd.DataFrame(index=pd.Index(obs.index[is_mouse], name="obs_name"))
for name in names:
    values = np.full(len(obs), np.nan)
    values[is_human] = cosine[name].to_numpy(float)
    transferred[name] = transfer(connectivities, values, is_human)[is_mouse]

print(f"mouse cells {len(transferred):,} | without a human neighbor {int(transferred[TARGET].isna().sum())}")

## Comparison with the final objects

The final objects store the Stro-7 values of the run of 29 October 2026. This run must give the same cells, the same
missing values and the same numbers within `TOLERANCE`; float32 sums can differ in their last digits between machines.

In [ ]:
def compare(computed, stored, label):
    """Largest |computed - stored|, after checking that both cover the same cells and agree on which are NaN."""
    assert set(computed.index) == set(stored.index), f"{label}: the cells differ"
    computed = computed.reindex(stored.index)
    assert computed.isna().equals(stored.isna()), f"{label}: the missing values differ"
    largest = float(np.nanmax(np.abs(computed.to_numpy(float) - stored.to_numpy(float))))
    print(f"{label}: {len(stored):,} cells, max |recomputed - stored| {largest:.2g}")
    assert largest <= TOLERANCE, f"{label}: differs from the stored values by {largest:.2g}"


if HUMAN_OBJECT is not None:
    stored = read_obs_column(HUMAN_OBJECT, "stro7_cosine")
    compare(cosine[TARGET], stored, "human stro7_cosine")

if MOUSE_OBJECT is not None:
    stored = read_obs_column(MOUSE_OBJECT, "stro7_cosine_transferred")
    compare(transferred[TARGET], stored, "mouse stro7_cosine_transferred")

## Outputs

`stro7_cosine_human.csv` (one row per human nucleus) and `stro7_cosine_mouse.csv` (one row per mouse cell), each with
the 11 subtype columns, `Stro-5` to `Stro-6`. Nothing is written when `OUT_DIR` is None, and nothing is replaced.

In [ ]:
if OUT_DIR is not None:
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    for table, filename in ((cosine, "stro7_cosine_human.csv"), (transferred, "stro7_cosine_mouse.csv")):
        path = out_dir / filename
        if path.exists():
            raise FileExistsError(f"{path} exists; choose a new OUT_DIR")
        table.to_csv(path)
        print("wrote", path)

print(f"human nuclei {len(cosine):,} | mouse cells {len(transferred):,}, "
      f"{int(transferred[TARGET].notna().sum()):,} with a value | "
      f"median Stro-7 cosine: human {cosine[TARGET].median():.4f}, mouse {transferred[TARGET].median():.4f}")